In [49]:
# mike babb
# created: 2026 08 23
# updated: 2026 09 23
# find five words with 25 different letters
# example: ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']

In [50]:
# standard
import math
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [51]:
import pandas as pd
import numpy as np

In [52]:
# custom
from utils import *
import _run_constants as rc

In [53]:
# CREATE A TEST VARIABLE
use_test = False

# LOAD DATA

In [54]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [55]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## DEMONSTRATE BITWISE OPERATIONS

In [56]:
vibex = byte_encode_words('vibex')
glyph = byte_encode_words('glyph')
muntz = byte_encode_words('muntz')
dwarf = byte_encode_words('dwarf')
jocks = byte_encode_words('jocks')
cramp = byte_encode_words('cramp')

In [57]:
# this is equal to zero - no letters reused
(vibex | glyph | muntz | dwarf) & jocks 

0

In [58]:
# this is not equal to zero because letters are reused
((vibex | glyph) | muntz | dwarf) & cramp

167937

In [59]:
# order of operations for bitwise operations
(vibex | glyph) & (muntz | dwarf) 

0

In [60]:
vibex | glyph | muntz | dwarf | jocks

67043327

In [61]:
# same output as above
byte_encode_words('vibexglyphmuntzdwarfjocks')

67043327

# BUILD LEVEL 2 BY COMBINING TWO BYTE ENCODED WORDS

In [62]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common, compute the bitwise or to add the words together
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])
l2_df.shape

(3213696, 3)

# BUILD LEVELS 4 AND 5 BY COMBINING TWO ITEMS FROM THE L2 LIST  
# COMPARE THAT WITH THE WORD BYTE ARRAY ONE MORE TIME

In [63]:
# get words from bytes
l2_df['w1'] = l2_df['w1b'].map(word_byte_to_word_dict)
l2_df['w2'] = l2_df['w2b'].map(word_byte_to_word_dict)

In [64]:
w_l2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [65]:
l2_all = w_l2_df['l2'].to_numpy(dtype = np.int32)

In [66]:
# let's just use the word jocks
if use_test:
    w_l2_df = w_l2_df.loc[(w_l2_df['w1'] == 'jocks') |
                                (w_l2_df['w2'] == 'jocks'), :].reset_index(drop = True)

In [67]:
w_l2_df = w_l2_df.drop(labels = ['w1', 'w2'], axis = 1)
l2_df = l2_df.drop(labels = ['w1', 'w2'], axis = 1)

In [68]:
w_l2_df.shape

(640023, 3)

In [69]:
w_l2_df['l2'].unique().shape

(640023,)

In [70]:
l2_all.shape

(640023,)

In [71]:
# we are going to make a lot of comparisons
print('The full set of l2 - duplicated l2:', l2_df.shape[0], l2_df.shape[0] ** 2)
print('The unique l2:', l2_df['l2'].unique().shape[0],  l2_df['l2'].unique().shape[0]** 2)
# but, we'll be clever about this and compare each item from the l2_list
# against the whole l2_list using array operations. 


The full set of l2 - duplicated l2: 3213696 10327841980416
The unique l2: 640023 409629440529


# COMPUTE THE COMBINATIONS

In [72]:
w_l2_df.shape

(640023, 3)

In [73]:
l2_df.head()

,w1b,w2b,l2
0,20491,264468,284959
1,20491,532756,553247
2,20491,788500,808991
3,20491,794644,815135
4,20491,1114388,1134879


In [74]:
l2_all.shape

(640023,)

In [75]:
w_l2_df.head()

,w1b,w2b,l2
0,20491,264468,284959
1,20491,532756,553247
2,20491,788500,808991
3,20491,794644,815135
4,20491,1114388,1134879


In [76]:
w_l2_df.shape

(640023, 3)

In [77]:
l2_df.shape

(3213696, 3)

In [78]:
# compute all possible pairs
start_pos = 0
total_output = np.zeros(shape = (100_000_000, 5), dtype = np.int32)
for i_row, row in w_l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row
    # print(w1b, w2b, l2)

    # compare the current l2 to all l2 - this will find all instances
    # a value of zero indicates that there are no letters in common
    # indexer for l2, l3, and l4
    # bitwise and to identify two l2 that do not have a letter in common
    positional_idx_l2l3l4 = (l2_all & l2) == 0 
    #print(positional_idx_l2l3l4.shape)
    # TODO: fix this!
    if positional_idx_l2l3l4.any():

        # these are l2 words with different letters the the l2 above. 
        # this is effectively l4
        # this is now 20 different letters
        output_array_w3bw4b = l2_all[positional_idx_l2l3l4]
        # print(output_array_w3bw4b.shape)
        
        # l2, l3, l4 accumulated letters
        output_array_l2l3l4 = output_array_w3bw4b | l2        
        
        # check against the word_byte_array for w5b        
        for w3bw4b, l2l3l4 in zip(output_array_w3bw4b, output_array_l2l3l4):
            # w3bw4b: this is the other l2. l2 and w3bw4b have no letters in common. 20 letters.
            # l2l3l4: this the combined l2 and l4 stuff. Through bitwise or operations. 

            # create an indexer over bitwise and
            positional_idx_l2l3l4l5 = (word_byte_array & l2l3l4) == 0

            if positional_idx_l2l3l4l5.any(): 
                # the indexer has at least one True value.
                
                # output_array_l2l3l4l5 is the list of word(s) that have letters
                # that do not match the other letters. In other words, this is 
                # the final five letters not in the group of twenty
                output_array_l2l3l4l5 = word_byte_array[positional_idx_l2l3l4l5]                
                # print(output_array_l2l3l4l5.shape)

                # count!
                n_rows_l2l3l4l5 = output_array_l2l3l4l5.shape[0]

                # create a temporary matrix to hold the output
                temp_output = np.zeros(shape = (n_rows_l2l3l4l5, 5), dtype = np.int32)
                # word 1
                temp_output[:, 0] = w1b
                # word 2
                temp_output[:, 1] = w2b
                # the bitwise or on w1 and w2
                temp_output[:, 2] = l2

                # calculate w3b and w4b
                # this is the 'other' w1b and w2b values - w3b and w4b, effectively.
                temp_output[:, 3] = w3bw4b
                # the final word
                temp_output[:, 4] = output_array_l2l3l4l5
                
                # update the total output with the temporary list
                total_output[start_pos:start_pos + n_rows_l2l3l4l5, :] = temp_output

                # the counter
                start_pos += n_rows_l2l3l4l5
                # print(start_pos) 
    
    if i_row % 1000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row)   

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000
113000
114000
115000
116000
117000
118000
119000
120000
121000
122000
123000
124000
125000
126000
127000
128000
129000
130000
131000
132000
133000
134000
135000
136000
137000
138000
139000
140000
141000
142000
143000
144000
145000
146000
147000
148000
149000
150000
151000
152000
153000
154000
155000
156000
157000
158000


# CREATE AND SHAPE THE OUTPUT

In [79]:
output = total_output[:start_pos]

In [80]:
# turn it into a dataframe
output_df = pd.DataFrame(data = output, columns = ['w1b', 'w2b', 'l2',  'l3l4', 'w5b'])
output_df.shape

(11008, 5)

In [81]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b
0,16912387,8914984,25827371,37013012,4202944
1,16912387,8914984,25827371,5547460,35668496
2,16912387,8914984,25827371,37020240,4195716
3,16912387,8914984,25827371,39864212,1351744
4,16912387,8914984,25827371,39871440,1344516


# JOIN TO GET THE W3B AND THE W4B

In [82]:
l3l4_df = l2_df[['w1b', 'w2b', 'l2']].copy()
l3l4_df.columns = ['w3b', 'w4b', 'l3l4',]

In [83]:
l3l4_df.shape

(3213696, 3)

In [84]:
output_df = pd.merge(left = output_df, right = l3l4_df)

In [85]:
output_df.shape

(13548, 7)

In [86]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b,w3b,w4b
0,16912387,8914984,25827371,37013012,4202944,1344516,35668496
1,16912387,8914984,25827371,5547460,35668496,1344516,4202944
2,16912387,8914984,25827371,5547460,35668496,1351744,4195716
3,16912387,8914984,25827371,37020240,4195716,1351744,35668496
4,16912387,8914984,25827371,39864212,1351744,35668496,4195716


In [87]:
# reorder...
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b']
output_df = output_df[col_names].copy()

In [88]:
# get words!
for ii in range(1, 6):
    bcn = f"w{ii}b"
    cn = f"w{ii}"
    output_df[cn] = output_df[bcn].map(word_byte_to_word_dict)

In [89]:
# count the remainder letter
lc_set = set(ascii_lowercase)
def get_remainder_letter(row):
    my_set = set()
    for cn in ['w1', 'w2', 'w3', 'w4', 'w5']:
        my_set.update(row[cn])

    return ''.join(lc_set.difference(my_set))

output_df['remaining_letter'] = output_df.apply(get_remainder_letter, axis = 1)

In [90]:
# get the word group
def get_remainder_letter(row):
    my_set = set()
    for cn in ['w1', 'w2', 'w3', 'w4', 'w5']:
        my_set.add(row[cn])

    return ' '.join(sorted(my_set))

output_df['word_group'] = output_df.apply(get_remainder_letter, axis = 1)

In [91]:
# count unique words - JUST TO VERIFY
col_names = ['w1', 'w2', 'w3', 'w4', 'w5']
output_df['n_unique_words'] = output_df[col_names].apply(lambda x: len(set(x)), axis = 1)

In [92]:
# add the words - ALSO TO VERIFY
output_df['bitwise_or'] = 0
output_df['bitwise_and'] = 0
for cn_idx in range(1, 6):
    b_cn = f"w{cn_idx}b"
    w_cn = f"w{cn_idx}"
    output_df[w_cn] = output_df[b_cn].map(word_byte_to_word_dict)
    output_df['bitwise_and'] = output_df['bitwise_and'] & output_df[b_cn]
    output_df['bitwise_or'] = output_df['bitwise_or'] | output_df[b_cn]


In [93]:
output_df.head()

,w1b,w2b,w3b,w4b,w5b,w1,w2,w3,w4,w5,remaining_letter,word_group,n_unique_words,bitwise_or,bitwise_and
0,16912387,8914984,1344516,35668496,4202944,ambry,fldxt,pucks,vejoz,whing,q,ambry fldxt pucks vejoz whing,5,67043327,0
1,16912387,8914984,1344516,4202944,35668496,ambry,fldxt,pucks,whing,vejoz,q,ambry fldxt pucks vejoz whing,5,67043327,0
2,16912387,8914984,1351744,4195716,35668496,ambry,fldxt,pungs,whick,vejoz,q,ambry fldxt pungs vejoz whick,5,67043327,0
3,16912387,8914984,1351744,35668496,4195716,ambry,fldxt,pungs,vejoz,whick,q,ambry fldxt pungs vejoz whick,5,67043327,0
4,16912387,8914984,35668496,4195716,1351744,ambry,fldxt,vejoz,whick,pungs,q,ambry fldxt pungs vejoz whick,5,67043327,0


In [94]:
output_df['word_group'].value_counts()

word_group
ambry fldxt pucks vejoz whing    30
bumps chivw fldxt gazon jerky    30
brugh campy fldxt swink vejoz    30
bucky flimp hdqrs twang vejoz    30
bumph fldxt gconv jerky swazi    30
                                 ..
fldxt jocks verby whump zigan    18
bumph fldxt jacky vower zings    18
fldxt jacko verby whump zings    18
fldxt ginzo jacks verby whump    18
bumps fldxt javer whick zygon    15
Name: count, Length: 538, dtype: int64

In [95]:
output_df['bitwise_or'].unique().shape

(11,)

# CREATE AND SAVE OUTPUT

In [96]:
output_df.to_excel(excel_writer='test.xlsx', index = False)